In [0]:
create or replace table production.supply_analytics.pricing_feature_audit_base as

WITH base_tours AS (
  SELECT DISTINCT
    dt.user_id as supplier_id,
    ss.segment,
    dt.tour_id,
    o.tour_option_id,
    dt.tour_title,
    o.tour_option_title,
    dtl.location_name,
    dt.activity_type,
    dt.category AS tour_category,
    dt.date_of_creation,
    dt.date_of_first_online,
    dt.is_top_attraction,
    CASE WHEN NOT(s.account_manager_email IS NULL 
              OR s.account_manager_email = 'nobounce@getyourguide.com' 
              OR (COALESCE(s.account_manager, 'Supplier Support')) = 'Supplier Support')  
     THEN 'Yes' 
     ELSE 'No' 
    END as is_managed,
    
    -- API INTEGRATION FLAGS
    iis.uses_prices_over_api,
    iis.uses_live_availability_and_price,
    iis.uses_pricing_scales_over_api
    
  FROM production.dwh.dim_tour dt
  LEFT JOIN production.dwh.dim_tour_option o ON dt.tour_id = o.tour_id
  LEFT JOIN production.dwh.dim_tour_location dtl ON dtl.location_id = dt.location_id AND dtl.tour_id = dt.tour_id
  LEFT JOIN production.db_mirror_dbz.inventory__inventory_settings iis ON o.tour_option_id = iis.tour_option_id
  left join production.dwh.dim_supplier s on dt.user_id = s.user_id
  LEFT JOIN production.supply_analytics.dim_supplier_segment ss ON dt.user_id = ss.user_id
  WHERE dt.gyg_status = 'active'
    AND dt.status = 'active'
    AND dt.is_online = true
    AND o.status = 'active'
),

pricing_config_date_coverage AS (
  SELECT 
    tour_option_id,
    pricing_id,
    COUNT(DISTINCT CAST(date_time AS DATE)) AS num_dates_covered,
    MIN(CAST(date_time AS DATE)) AS earliest_date,
    MAX(CAST(date_time AS DATE)) AS latest_date
  FROM db_mirror_dbz.inventory_generated__available_time_slot
  WHERE CAST(date_time AS DATE) BETWEEN CURRENT_DATE() AND DATE_ADD(CURRENT_DATE(), 365)
  GROUP BY tour_option_id, pricing_id
),

vacancy_date_coverage AS (
  SELECT 
    tour_option_id,
    pricing_id,
    COUNT(DISTINCT CAST(date_time AS DATE)) AS num_dates_with_vacancies,
    COUNT(DISTINCT CASE WHEN vacancies IS NOT NULL AND vacancies > 0 THEN CAST(date_time AS DATE) END) AS num_dates_with_available_vacancies,
    COUNT(DISTINCT CASE WHEN max_vacancies IS NOT NULL THEN CAST(date_time AS DATE) END) AS num_dates_with_capacity_configured
  FROM db_mirror_dbz.inventory_generated__available_time_slot
  WHERE CAST(date_time AS DATE) BETWEEN CURRENT_DATE() AND DATE_ADD(CURRENT_DATE(), 365)
  GROUP BY tour_option_id, pricing_id
),

latest_config_per_pricing AS (
  SELECT 
    pcdc.tour_option_id,
    pcdc.pricing_id,
    pcdc.num_dates_covered,
    pcdc.earliest_date,
    pcdc.latest_date,
    ats.from_price_without_deal,
    ats.currency_iso_code,
    ats.details_deserialized,
    ats.update_timestamp,
    ats.cut_off_time_seconds,
    ats.min_booking_participants,
    ats.max_booking_participants,
    ats.vacancies,
    ats.max_vacancies,
    ats.reserved_vacancies,
    ats.unavailable_reasons,
    
    ROW_NUMBER() OVER (
      PARTITION BY pcdc.tour_option_id, pcdc.pricing_id
      ORDER BY ats.update_timestamp DESC
    ) as rn
    
  FROM pricing_config_date_coverage pcdc
  INNER JOIN db_mirror_dbz.inventory_generated__available_time_slot ats
    ON pcdc.tour_option_id = ats.tour_option_id 
    AND pcdc.pricing_id = ats.pricing_id
  WHERE ats.details_deserialized IS NOT NULL
),

pricing_config_exploded AS (
  SELECT 
    tour_option_id,
    pricing_id,
    currency_iso_code,
    num_dates_covered,
    earliest_date,
    latest_date,
    from_price_without_deal,
    
    from_json(
      details_deserialized, 
      'struct<pricingCategories:struct<individual:array<struct<ticketCategory:string,minAge:int,maxAge:int,resolvedCommissionRate:decimal(10,3),independentlyBookable:boolean,bookingType:string,prices:array<struct<maxParticipants:int,retailPrice:decimal(10,2),netPrice:decimal(10,2)>>>>,group:array<struct<resolvedCommissionRate:decimal(10,3),isAdditionalPaxForGroup:boolean,prices:array<struct<maxParticipants:int,retailPrice:decimal(10,2),netPrice:decimal(10,2)>>>>,addons:array<struct<addonId:int,resolvedCommissionRate:decimal(10,3),prices:array<struct<maxParticipants:int,retailPrice:decimal(10,2),netPrice:decimal(10,2)>>>>>,group:array<struct<resolvedCommissionRate:decimal(10,3),isAdditionalPaxForGroup:boolean,prices:array<struct<maxParticipants:int,retailPrice:decimal(10,2),netPrice:decimal(10,2)>>>>,addons:array<struct<addonId:int,resolvedCommissionRate:decimal(10,3),prices:array<struct<maxParticipants:int,retailPrice:decimal(10,2),netPrice:decimal(10,2)>>>>>'
    ) as parsed_data,
    
    update_timestamp as config_snapshot_timestamp,
    cut_off_time_seconds,
    min_booking_participants,
    max_booking_participants,
    vacancies,
    max_vacancies,
    reserved_vacancies,
    unavailable_reasons
    
  FROM latest_config_per_pricing
  WHERE rn = 1
),
normalized_pricing AS (
  SELECT
    tour_option_id,
    pricing_id,
    currency_iso_code,
    num_dates_covered,
    earliest_date,
    latest_date,
    from_price_without_deal,
    
    COALESCE(parsed_data.pricingCategories.individual, ARRAY()) as individual_categories,
    COALESCE(parsed_data.pricingCategories.group, parsed_data.group, ARRAY()) as group_categories_raw,
    COALESCE(parsed_data.pricingCategories.addons, parsed_data.addons, ARRAY()) as addons_raw,
    
    config_snapshot_timestamp,
    cut_off_time_seconds,
    min_booking_participants,
    max_booking_participants,
    vacancies,
    max_vacancies,
    reserved_vacancies,
    unavailable_reasons
    
  FROM pricing_config_exploded
),

pricing_features_per_config AS (
  SELECT 
    tour_option_id,
    pricing_id,
    currency_iso_code,
    num_dates_covered,
    earliest_date,
    latest_date,
    from_price_without_deal,
    
    -- RAW CATEGORY DATA
    individual_categories,
    group_categories_raw,
    addons_raw,
    
    -- COUNTS
    SIZE(individual_categories) as num_individual_categories,
    SIZE(group_categories_raw) as num_group_categories,
    SIZE(addons_raw) as num_addons,
    
    -- TIER COUNTS
    AGGREGATE(individual_categories, 0, (acc, ind) -> acc + SIZE(ind.prices)) as total_individual_price_tiers,
    AGGREGATE(group_categories_raw, 0, (acc, grp) -> acc + SIZE(grp.prices)) as total_group_price_tiers,
    AGGREGATE(addons_raw, 0, (acc, addon) -> acc + SIZE(addon.prices)) as total_addon_price_tiers,
    
    -- SCALE PRICING FLAGS
    CASE 
      WHEN SIZE(FILTER(individual_categories, ind -> SIZE(ind.prices) > 1)) > 0 
      THEN 1 ELSE 0 
    END as has_scale_pricing_individual,
    
    CASE 
      WHEN SIZE(FILTER(group_categories_raw, grp -> SIZE(grp.prices) > 1)) > 0 
      THEN 1 ELSE 0 
    END as has_scale_pricing_group,
    
    CASE 
      WHEN SIZE(FILTER(addons_raw, addon -> SIZE(addon.prices) > 1)) > 0 
      THEN 1 ELSE 0 
    END as has_scale_pricing_addons,
    
    config_snapshot_timestamp,
    cut_off_time_seconds,
    min_booking_participants,
    max_booking_participants,
    vacancies,
    max_vacancies,
    reserved_vacancies,
    unavailable_reasons
    
  FROM normalized_pricing
)

SELECT 
  bt.supplier_id,
  bt.segment,
  bt.tour_id,
  bt.tour_option_id,
  pfc.pricing_id,
  bt.tour_title,
  bt.tour_option_title,
  bt.location_name,
  bt.activity_type,
  bt.tour_category,
  bt.date_of_creation,
  bt.date_of_first_online,
  bt.is_top_attraction,
  pfc.from_price_without_deal,
  bt.is_managed,
  
  -- API INTEGRATION FLAGS
  bt.uses_prices_over_api,
  bt.uses_live_availability_and_price,
  bt.uses_pricing_scales_over_api,
  
  -- DERIVED FLAG: External pricing (exclude from native pricing analysis)
  CASE 
    WHEN bt.uses_prices_over_api = 1 
      OR bt.uses_pricing_scales_over_api = 1 
    THEN 1 ELSE 0 
  END as uses_external_pricing,
  
  pfc.currency_iso_code,
  pfc.num_dates_covered AS pricing_dates_covered_next_12mo,
  pfc.earliest_date AS pricing_valid_from,
  pfc.latest_date AS pricing_valid_to,
  
  -- RAW PRICING CATEGORY DETAILS (JSON COLUMNS)
  pfc.individual_categories,
  pfc.group_categories_raw as group_categories,
  pfc.addons_raw as addons,
  
  -- PRICING FEATURES (COUNTS)
  pfc.num_individual_categories,
  pfc.num_group_categories,
  pfc.num_addons,
  
  CASE WHEN pfc.num_addons > 0 THEN 1 ELSE 0 END AS has_addons_configured,
  CASE WHEN pfc.num_individual_categories > 0 THEN 1 ELSE 0 END AS has_individual_pricing,
  CASE WHEN pfc.num_group_categories > 0 THEN 1 ELSE 0 END AS has_group_pricing,
  
  pfc.total_individual_price_tiers,
  pfc.total_group_price_tiers,
  pfc.total_addon_price_tiers,
  (pfc.total_individual_price_tiers + pfc.total_group_price_tiers + pfc.total_addon_price_tiers) AS total_price_tiers,
  
  pfc.has_scale_pricing_individual,
  pfc.has_scale_pricing_group,
  pfc.has_scale_pricing_addons,
  
  CASE 
    WHEN pfc.has_scale_pricing_individual = 1 
      OR pfc.has_scale_pricing_group = 1 
      OR pfc.has_scale_pricing_addons = 1 
    THEN 1 ELSE 0 
  END AS has_any_scale_pricing,
  
  -- OPERATIONAL FEATURES
  ROUND(pfc.cut_off_time_seconds / 3600.0, 1) as cutoff_hours,
  CASE WHEN pfc.cut_off_time_seconds IS NOT NULL AND pfc.cut_off_time_seconds > 0 THEN 1 ELSE 0 END as has_cutoff_configured,
  
  pfc.min_booking_participants,
  pfc.max_booking_participants,
  CASE WHEN pfc.min_booking_participants IS NOT NULL AND pfc.min_booking_participants > 1 THEN 1 ELSE 0 END as has_min_participant_requirement,
  CASE WHEN pfc.max_booking_participants IS NOT NULL THEN 1 ELSE 0 END as has_max_participant_limit,
  
  -- VACANCY COVERAGE
  vdc.num_dates_with_vacancies,
  vdc.num_dates_with_available_vacancies,
  vdc.num_dates_with_capacity_configured,
  
  CASE 
    WHEN pfc.num_dates_covered > 0 
    THEN ROUND(100.0 * vdc.num_dates_with_vacancies / pfc.num_dates_covered, 1)
    ELSE NULL
  END as pct_dates_with_vacancy_info,
  
  pfc.vacancies as sample_vacancies,
  pfc.max_vacancies as sample_max_vacancies,
  pfc.reserved_vacancies as sample_reserved_vacancies,
  
  CASE WHEN pfc.max_vacancies IS NOT NULL THEN 1 ELSE 0 END as has_capacity_limits,
  CASE WHEN pfc.vacancies IS NOT NULL AND pfc.vacancies > 0 THEN 1 ELSE 0 END as has_available_slots,
  CASE WHEN pfc.unavailable_reasons IS NOT NULL AND pfc.unavailable_reasons != '' THEN 1 ELSE 0 END as has_unavailable_slots,
  
  pfc.config_snapshot_timestamp

FROM base_tours bt
INNER JOIN pricing_features_per_config pfc ON bt.tour_option_id = pfc.tour_option_id
LEFT JOIN vacancy_date_coverage vdc ON pfc.tour_option_id = vdc.tour_option_id AND pfc.pricing_id = vdc.pricing_id

ORDER BY bt.tour_id, bt.tour_option_id, pfc.pricing_id;
